In [171]:
import pandas as pd
import numpy as np
import matplotlib as ptl

# **1.1 — Load Data**

In [172]:
featurs_df=pd.read_csv("../data/generated_v2/releases/v2_20261028/account_month_features.csv")
labels_df=pd.read_csv("../data/generated_v2/releases/v2_20261028/evaluation_labels.csv")

# **1.2 — Data Integrity Validation**

In [173]:
featurs_df.duplicated(subset=["account_id", "month"]).sum()

np.int64(0)

In [174]:
labels_df.duplicated(subset=["account_id","month"]).sum()

np.int64(0)

In [175]:
merged=featurs_df.merge(labels_df,on=["account_id","month"], how="outer",indicator=True)

In [176]:
merged["_merge"].value_counts()

_merge
both          6000
left_only        0
right_only       0
Name: count, dtype: int64

In [177]:
merged["_merge"].eq("both").all()

np.True_

# **1.3 — Define Eligible Features**

In [178]:
initial_eligible_features=[
    col for col in featurs_df.columns
    if col not in ["account_id", "month"]
    and not col.endswith("_robust_z_past") 
]
print(initial_eligible_features)
print("Eligible feature count:" , len(initial_eligible_features))

['txn_count', 'unique_counterparties', 'inflow_sum', 'outflow_sum', 'inflow_avg', 'outflow_avg', 'net_flow', 'inflow_outflow_ratio', 'zero_outflow_flag', 'flow_concentration', 'txn_count_change_vs_prev', 'unique_counterparties_change_vs_prev', 'inflow_sum_change_vs_prev', 'outflow_sum_change_vs_prev', 'net_flow_change_vs_prev']
Eligible feature count: 15


# **1.4 — Define Feature Groups**

In [179]:
featurs_df.columns.tolist()

['account_id',
 'month',
 'txn_count',
 'unique_counterparties',
 'inflow_sum',
 'outflow_sum',
 'inflow_avg',
 'outflow_avg',
 'net_flow',
 'inflow_outflow_ratio',
 'zero_outflow_flag',
 'flow_concentration',
 'txn_count_change_vs_prev',
 'txn_count_robust_z_past',
 'unique_counterparties_change_vs_prev',
 'unique_counterparties_robust_z_past',
 'inflow_sum_change_vs_prev',
 'inflow_sum_robust_z_past',
 'outflow_sum_change_vs_prev',
 'outflow_sum_robust_z_past',
 'net_flow_change_vs_prev',
 'net_flow_robust_z_past']

In [180]:
behavioral_features = [
    "txn_count",
    "unique_counterparties",
    "txn_count_change_vs_prev",
    "unique_counterparties_change_vs_prev"
]
fund_flow_extended_features = [
    "inflow_sum",
    "outflow_sum",
    "inflow_avg",
    "outflow_avg",
    "net_flow",
    "inflow_outflow_ratio",
    "zero_outflow_flag",
    "flow_concentration",
    "inflow_sum_change_vs_prev",
    "outflow_sum_change_vs_prev",
    "net_flow_change_vs_prev"
]
fund_flow_core_features = [
    "inflow_sum",
    "outflow_sum",
    "inflow_avg",
    "outflow_avg",
    "net_flow",
    "inflow_outflow_ratio",
    "flow_concentration"
]

combined_core_features = behavioral_features + fund_flow_core_features

feature_groups = {
    "behavioral": behavioral_features,
    "fund_flow_extended": fund_flow_extended_features,
    "fund_flow_core": fund_flow_core_features,
    "combined_core": combined_core_features
}

In [ ]:
for group,features  in feature_groups.items():
    print(group, len(features))
    

behavioral 4
fund_flow_extended 11
fund_flow_core 7
combined_core 11


In [182]:
assert all(
    set(features).issubset(initial_eligible_features)
    for features in feature_groups.values()
)

## **1.5 — Missing Value Analysis**

In [183]:
featurs_df[combined_core_features].isna().sum()

txn_count                                  0
unique_counterparties                      0
txn_count_change_vs_prev                1000
unique_counterparties_change_vs_prev    1000
inflow_sum                                 0
outflow_sum                                0
inflow_avg                                23
outflow_avg                               34
net_flow                                   0
inflow_outflow_ratio                      34
flow_concentration                         0
dtype: int64

In [184]:
featurs_df.groupby("month")[combined_core_features].apply(
    lambda x:x.isna().sum()
)

,txn_count,unique_counterparties,txn_count_change_vs_prev,unique_counterparties_change_vs_prev,inflow_sum,outflow_sum,inflow_avg,outflow_avg,net_flow,inflow_outflow_ratio,flow_concentration
month,,,,,,,,,,,
1,0,0,1000,1000,0,0,0,6,0,6,0
2,0,0,0,0,0,0,2,3,0,3,0
3,0,0,0,0,0,0,5,8,0,8,0
4,0,0,0,0,0,0,4,5,0,5,0
5,0,0,0,0,0,0,4,4,0,4,0
6,0,0,0,0,0,0,8,8,0,8,0
